In [0]:
%pip install pdfplumber

In [0]:
pip install PyMuPDF

In [0]:
########################################
## Import Libraries
########################################

import fitz 
import base64
import requests
import json
import json
import re
import pandas as pd
from pyspark.sql.functions import lit, current_timestamp
from databricks.sdk import WorkspaceClient

wkc = WorkspaceClient()

In [0]:
########################################
## Huge Prompt for Deepseek
########################################

prompt= """
Toma esa imagen y devuelve el JSON como un JSON valido 
La respuesta debe ser únicamente JSON válido, con la siguiente estructura exacta:

1. FORMATO DE SALIDA (JSON)
La respuesta debe ser únicamente JSON válido, con la siguiente estructura exacta:

	json
	[
	 {
		"nombre_mercado": "null",
		"nombre_producto": "null",
		"unidad_medida": "null",
		"inicio_fecha_precios": "null",
		"fin_fecha_precios": "null",
		"precio_inicio": "null",
		"precio_fin": "null",
		"error": "null"
	 }
	]
Regla crítica: Todos los valores deben ir como strings (entre comillas dobles), incluyendo "null" cuando no haya dato.

Toma las siguientes consideraciones como reglas a aplicar al momento de construir el JSON válido

2. REGLAS DE EXTRACCIÓN DE CAMPOS
2.1 nombre_mercado
	Tomar el texto que está dentro del rectángulo amarillo (título del mercado/feria/supermercado).

	Normalizar contra la lista oficial de mercados comunes (minúsculas, con tildes correctas).

	Si el texto no coincide con la lista, mantener el texto tal cual se leyó.

	Si no se puede leer, colocar "null" y reportar en "error".

	Lista oficial de mercados normalizados:

	- mercado zonal belén, comayagüela
	- ahorro ferias del pueblo
	- feria del agricultor y el artesano
	- supermercados tegucigalpa
	- supermercados san pedro sula
	- mercado el rápido san pedro sula
	- feria agropecuaria de villanueva
	- mercado san isidro, las americas y colón

2.2 nombre_producto
	Usar el catálogo estándar de productos según la posición de la fila en la tabla.
	Si la posición no coincide con el catálogo, intentar leer el texto directamente.
	Si no se puede leer, colocar "null" y reportar en "error".

2.3 unidad_medida
	Usar el catálogo estándar de unidades según la posición de la fila.
	Si no se puede determinar, colocar "null" y reportar en "error".

2.4 inicio_fecha_precios y fin_fecha_precios
	Buscar el texto "( Semana del ...)"* o "Semana del ..." en la parte superior de las columnas.
	Dividirlo en dos fechas con formato DD-MM-AAAA:
	inicio_fecha_precios: primer valor de la semana
	fin_fecha_precios: segundo valor de la semana

2.5 precio_inicio
	Tomar el valor de la primera columna de "Precios L" (o "Precio L. [fecha inicio]").

2.6 precio_fin
	Tomar el valor de la segunda columna de "Precios L" (o "Precio L. [fecha fin]").

2.7 error
	Si no hay problemas: "null"
	Si hay problemas de lectura: indicar número de página + nombre del documento.
	Formato exacto: "pag X - nombre_del_archivo.pdf"

	3. REGLAS DE PROCESAMIENTO DE PÁGINAS Y TABLAS
	Procesar todas las páginas del archivo, sin excepción, desde la página 1 hasta la última del archivo.
	Incluir todas las tablas que aparezcan en cada página, aunque estén parcialmente cortadas, borrosas o incompletas.
	Si una tabla no se puede leer en absoluto:
	Crear una sola entrada JSON para esa tabla.
	Poner "null" en todos los campos.
	Indicar en "error" el número de página + nombre del documento.
	Si una tabla se puede leer parcialmente:
	Incluir una entrada JSON por cada fila que se pueda leer.
	En las filas donde falte algún valor, poner "null" en ese campo.
	En el campo "error" de esa fila, indicar el número de página + nombre del documento.
	Nunca omitir una tabla por estar borrosa, cortada o con texto corrupto.
	Si una página contiene solo ruido o texto sin estructura de tabla:
	No se genera entrada de tabla.
	Se debe reportar con una entrada JSON con todos los campos "null" y en "error" el número de página + nombre del documento.
	No agrupar ni resumir: cada fila de cada tabla debe ser un objeto JSON independiente.
	Si no se puede leer un valor de una fila de la tabla, colocar "null" en ese campo (no omitirlo) y reportar en "error" el número de página + nombre del documento.

4. CATÁLOGO ESTÁNDAR DE PRODUCTOS Y UNIDADES
	Usar este catálogo según la posición de la fila en la tabla (1 a 30):

	#	Producto	Unidad
	1	Tajo de Res	libra
	2	Costilla de Res Regular	libra
	3	Costilla de Cerdo Regular	libra
	4	Pollo Entero Congelado sin Menudos	libra
	5	Pescado Blanco	libra
	6	Leche Integra en Polvo	360 g
	7	Leche pasteurizada fluida en bolsa	0.946l
	8	Mantequilla	libra
	9	Queso Blanco Fresco	libra
	10	Huevo Mediano	cartón
	11	Cebolla Amarilla	libra
	12	Tomate Pera	libra
	13	Papas	libra
	14	Yuca	libra
	15	Repollo	libra
	16	Plátano Maduro	unidad
	17	Naranja Dulce	unidad
	18	Banano Fresco Maduro	unidad
	19	Frijol Rojo a Granel	libra
	20	Arroz Clasificado a Granel	libra
	21	Espagueti	200 g
	22	Azúcar Blanca	libra
	23	Café Molido	16 oz
	24	Salsa de Tomate	400 g
	25	Aceite Vegetal	443 ml
	26	Manteca Comestible de Origen Vegetal	libra
	27	Sal común de Mesa Yodada	227 g
	28	Tortilla de Maíz	unidad
	29	Pan Molde Blanco	540 g
	30	Jugo de Naranja	500ml
	Regla adicional: Cuando una tabla no tenga exactamente 30 filas, usar la posición de cada fila para asignar el nombre y unidad correctos según este catálogo, en lugar de depender del texto ilegible.

5. CORRECCIONES COMUNES DE LECTURA (APLICAR AUTOMÁTICAMENTE)
5.1 Productos
	Texto ilegible	Corrección
	Cotilla	Costilla
	Mantegulla	Mantequilla
	Horro Mediano / Horno Mediano / Hervo Mediano	Huevo Mediano
	Pilatano Maduro / Píñano Maduro	Plátano Maduro
	Fitjol Rojo / Frijol Rojo a Granad	Frijol Rojo a Granel
	Acerle Vegetal	Aceite Vegetal
	Leche Pasteurizada India en Bolos	Leche pasteurizada fluida en bolsa
	Leche de Menudo	Manteca Comestible de Origen Vegetal
	Tomate Pecа / Teca	Tomate Pera / Yuca
	Anicar Blanca	Azúcar Blanca
	Pan Molido Blanco	Pan Molde Blanco
	Jugo de Naranja (Bolsa 500 ml)	Jugo de Naranja (500ml)

5.2 Mercados
	Texto ilegible	Corrección
	Mercado Zonal Belén, Comayaguela	mercado zonal belén, comayagüela
	Ahorro Ferias del Pueblo / Ahorro Feria del Pueblo	ahorro ferias del pueblo
	Feria del Agricultor y El Artesano	feria del agricultor y el artesano
	Supermercados Tegucigalpa / Supermercados Tequiegala	supermercados tegucigalpa
	Supermercado San Pedro Sula / Supermercados San Pedro Sula	supermercados san pedro sula
	Mercado El Rápido San Pedro Sula / Mercado El Rapido San Pedro Sula	mercado el rápido san pedro sula
	Feria Agropecuaria de Villanueva / Feria Agropecuaria Villanueva	feria agropecuaria de villanueva
	Mercados San Isidro, las Americas y Colón	mercado san isidro, las americas y colón
	
6. CHECKLIST FINAL ANTES DE ENTREGAR
	-¿Se procesaron todas las páginas del PDF?
	-¿Se incluyó al menos una entrada JSON por cada tabla encontrada?
	-¿Todos los valores están entre comillas dobles (strings)?
	-¿Los campos sin dato tienen "null" y su "error" correspondiente?
	-¿Los nombres de mercado están normalizados según la lista oficial?
	-¿Los nombres de producto y unidades siguen el catálogo estándar?
	-¿Las fechas están en formato DD-MM-AAAA?
	-¿El campo "error" usa el formato "pag X - nombre_del_archivo.pdf"?
	-¿No se agrupó ni resumió ninguna fila?
	-¿El JSON es válido y parseable?
 
 Asegúrate de completar todo el JSON de esta página sin truncarlo. Si es muy largo, ajusta el formato para que sea más compacto.

"""

In [0]:
API_KEY = dbutils.secrets.get(scope = "deepApiKey", key = "ApiKey")

PDF_ROUTE='/Workspace/Users/jarodriguezv91@gmail.com/Product_Management_D_E_Portfolio/Products_Management/src/PDF_Files'

column_order = [
    "nombre_mercado",
    "nombre_producto",
    "unidad_medida",
    "inicio_fecha_precios",
    "fin_fecha_precios",
    "precio_inicio",
    "precio_fin",
    "error",
    ]   


#RUTA_PDF='/Workspace/Users/jarodriguezv91@gmail.com/Product_Management_D_E_Portfolio/Products_Management/src/PDF_Files/2025_1_enero.pdf'

In [0]:
########################################
## Partition PDF into Images
########################################

def pdf_a_imagenes_base64_divididas(ruta_pdf, dpi=200):
    """
    Convierte cada página del PDF en dos imágenes (Mitad Superior y Mitad Inferior)
    en formato PNG codificadas en base64 para evitar superar el límite de tokens de la API.
    """
    doc = fitz.open(ruta_pdf)
    imagenes = []

    for i, pagina in enumerate(doc):
        rect = pagina.rect  # Obtiene las dimensiones (ancho, alto) de la página
        ancho = rect.width
        alto = rect.height

        # Definimos las dos mitades (rectángulos: x0, y0, x1, y1)
        # Mitad Superior
        rect_arriba = fitz.Rect(0, 0, ancho, alto / 2)
        # Mitad Inferior
        rect_abajo = fitz.Rect(0, alto / 2, ancho, alto)

        mitades = [("parte_1_superior", rect_arriba), ("parte_2_inferior", rect_abajo)]

        for nombre_parte, rect_mitad in mitades:
            # Renderizamos solo el rectángulo de la mitad especificada
            pix = pagina.get_pixmap(dpi=dpi, clip=rect_mitad)
            img_bytes = pix.tobytes("png")
            b64 = base64.b64encode(img_bytes).decode("utf-8")

            # Guardamos la página con un identificador (ej: página 1-parte_1_superior)
            imagenes.append(
                {"pagina": f"{i + 1}-{nombre_parte}", "base64": b64, "formato": "png"}
            )

    return imagenes


def pdf_a_imagenes_base64(ruta_pdf, dpi=200):
    """Convierte cada página del PDF a PNG en base64."""
    doc = fitz.open(ruta_pdf)
    imagenes = []
    for i, pagina in enumerate(doc):
        pix = pagina.get_pixmap(dpi=dpi)
        img_bytes = pix.tobytes("png")
        b64 = base64.b64encode(img_bytes).decode("utf-8")
        imagenes.append({"pagina": i + 1, "base64": b64, "formato": "png"})
    return imagenes

In [0]:
########################################
## Consume DeepSeek
########################################

def consultar_deepseek_vision(api_key, prompt, imagenes_b64):
    url = "https://api.deepseek.com/v1/chat/completions"
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
    # Construir contenido multimodal
    content = [{"type": "text", "text": prompt}]
    for img in imagenes_b64:
        content.append(
            {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{img}"}}
        )

    payload = {
        "model": "deepseek-reasoner",
        "messages": [{"role": "user", "content": content}],
        "temperature": 0.0,
        "max_tokens": 16000
    }
    resp = requests.post(url, headers=headers, json=payload, timeout=180)
    resp.raise_for_status()

    return resp.json()["choices"][0]["message"]["content"]

In [0]:
########################################
## Read File Container
########################################

def readFileContainer():
    # List files in a workspace directory
    for file in wkc.workspace.list(PDF_ROUTE):
        file_name = file.path
        resultados = []
        imagenes=''
        query = f"SELECT 1 FROM products.landing.docs_name WHERE docs_name = '{file_name}'"

        df = spark.sql(query)

        if df.isEmpty():
# Insert the doc name into control table            
            query = f"INSERT INTO products.landing.docs_name(docs_name) VALUES ('{file_name}')"
            df = spark.sql(query)

# Divinding the document into images
            imagenes = pdf_a_imagenes_base64(file_name)
            for img in imagenes:  
                resp = consultar_deepseek_vision(
                    API_KEY,
                    prompt,
                    [img["base64"]]
                )
                resultados.append({"pagina": img["pagina"], "datos": resp})
            ## Agregado 1123pm
            format_json_response(resultados)
            ########
            print("The file was read and processed")
            
        else:
            print("The file was already processed")

    


In [0]:
########################################
## Formating JSON into Requiered Dataset
########################################
def format_json_response(json_response):
    lista_productos_global = []

    for pagina_item in json_response:
        num_pagina = pagina_item.get("pagina", "desconocida")
        datos_str = pagina_item.get("datos", "")

        if not datos_str or datos_str == "null":
            continue

        # 1. Limpiar marcas de markdown si las tiene
        datos_limpios = datos_str.replace("```json", "").replace("```", "").strip()

        try:
            # Intentar parsear el JSON completo directamente (si la página no está truncada)
            productos_pagina = json.loads(datos_limpios)
        except json.JSONDecodeError:
            # Si está truncado, extraemos de forma segura todos los objetos {} que estén completos
            #print(
             #   f"Aviso: La página {num_pagina} estaba truncada. Extrayendo registros parciales válidos..."
            #)

            productos_pagina = []
            # Buscar bloques de llaves que representan cada producto individual
            matches = re.finditer(r"\{([^}]+)\}", datos_limpios)
            for match in matches:
                obj_str = "{" + match.group(1) + "}"
                try:
                    obj = json.loads(obj_str)
                    productos_pagina.append(obj)
                except json.JSONDecodeError:
                    # Descarta objetos individuales que también hayan quedado cortados al final
                    pass

        # 2. Normalizar y acumular los registros obtenidos
        for producto in productos_pagina:
            prod_normalizado = {
                "nombre_mercado": producto.get("nombre_mercado")
                if producto.get("nombre_mercado") not in [None, "null"]
                else None,
                "nombre_producto": producto.get("nombre_producto")
                if producto.get("nombre_producto") not in [None, "null"]
                else None,
                "unidad_medida": producto.get("unidad_medida")
                if producto.get("unidad_medida") not in [None, "null"]
                else None,
                "inicio_fecha_precios": producto.get("inicio_fecha_precios")
                if producto.get("inicio_fecha_precios") not in [None, "null"]
                else None,
                "fin_fecha_precios": producto.get("fin_fecha_precios")
                if producto.get("fin_fecha_precios") not in [None, "null"]
                else None,
                "precio_inicio": producto.get("precio_inicio")
                if producto.get("precio_inicio") not in [None, "null"]
                else None,
                "precio_fin": producto.get("precio_fin")
                if producto.get("precio_fin") not in [None, "null"]
                else None,
                "error": producto.get("error")
                if producto.get("error") not in [None, "null"]
                else None,
            }
            lista_productos_global.append(prod_normalizado)
    #return lista_productos_global
    
    
    ##agregado 1125pm
    write_to_delta(lista_productos_global)

    ########


In [0]:
# ==========================================
# CONVERT SPARK DATAFRAME TO LANDING TABLE
# ==========================================

def write_to_delta(df_json):

    df_pandas = pd.DataFrame(df_json)
    df_spark = spark.createDataFrame(df_pandas)
    df_product_list = df_spark.select(column_order)
    df_product_list_bronze = df_product_list.withColumn("ingesta_timestamp", current_timestamp())
    df_product_list_bronze.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("products.landing.product_list_markets_landing")

In [0]:
json_response=readFileContainer()

#format_json=format_json_response(json_response)

#write_to_delta(format_json)


In [0]:
%sql
select * from products.landing.docs_name

In [0]:
%sql
--Checking Landing table data 

select * from products.landing.product_list_markets_landing
